# Collaborative Filtering in Pure PyTorch

Collaborative filtering is one of the most successful approaches to recommendation systems. The idea: if users who liked the same movies as you also liked movie X, you'll probably like movie X too.

We don't need to know *anything* about the movies themselves — no genres, no actors, no descriptions. Just the pattern of who liked what. From that alone, the model discovers hidden structure.

This notebook builds a full recommendation system from scratch in PyTorch. No fastai, no high-level wrappers — just tensors, gradients, and a training loop. We'll go through these steps:

1. Load and explore movie rating data
2. Build intuition for *latent factors* — hidden qualities that describe taste
3. Connect the dots between one-hot encoding and embeddings
4. Train a dot product model, then improve it with biases and weight decay
5. Interpret what the model learned (which movies are universally loved? which are similar?)
6. Handle the cold start problem (new users with no history)
7. Build a neural network version that can learn nonlinear patterns

If you've gone through the fastai version of this notebook, think of this as the "what's actually happening under the hood" version. Same concepts, no abstractions hiding the machinery.

In [ ]:
# Setup - run this first
# If you're using the Colab extension in VS Code, upload the repo's data/ folder to the runtime:
#   Right-click the 'data' folder in VS Code -> "Upload to Colab Session"
from pathlib import Path
try:
    import google.colab
    DATA_PATH = Path('/content/data')
except ImportError:
    DATA_PATH = Path('../../../data')
print(f"Data path: {DATA_PATH}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

import urllib.request
import zipfile
import os
from pathlib import Path

# Use GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

## Downloading the Data

We'll use [MovieLens 100K](https://grouplens.org/datasets/movielens/) — 100,000 ratings from 943 users on 1,682 movies. Small enough to train quickly, large enough to learn real patterns.

In [ ]:
DATA_DIR = Path("../../../data/ml-100k")
if not DATA_DIR.exists():
    # Auto-download if not found
    import urllib.request, zipfile
    urllib.request.urlretrieve("https://files.grouplens.org/datasets/movielens/ml-100k.zip", "/tmp/ml-100k.zip")
    with zipfile.ZipFile("/tmp/ml-100k.zip") as z:
        z.extractall(".")
    DATA_DIR = Path("ml-100k")
    print("Downloaded MovieLens 100K")
else:
    print(f"Data already exists at {DATA_DIR}")

## Part 1: The Data

Imagine you have a giant spreadsheet. Each row is one rating: "User 196 gave Movie 242 a score of 3." That's it. No descriptions, no genres — just who rated what and how much.

The main file is `u.data` — tab-separated with columns: user ID, movie ID, rating (1-5), timestamp. Let's load it and see what we're working with.

In [ ]:
# Load the ratings
ratings = pd.read_csv(
    DATA_DIR / "u.data",
    delimiter="\t",
    header=None,
    names=["user", "movie", "rating", "timestamp"]
)

# Load movie titles for interpretation later
movies = pd.read_csv(
    DATA_DIR / "u.item",
    delimiter="|",
    encoding="latin-1",
    usecols=(0, 1),
    names=("movie", "title"),
    header=None
)

# Merge so we have titles alongside ratings
ratings = ratings.merge(movies)
print(f"{len(ratings):,} ratings from {ratings['user'].nunique()} users on {ratings['title'].nunique()} movies")
ratings.head()

If you imagine this data as a big grid — users as rows, movies as columns — most cells would be empty. The average user has rated about 106 movies out of 1,664. That's a **sparse** matrix: over 93% empty.

Our job is to fill in those blanks. If we can predict what User 42 would rate a movie they haven't seen yet, we can recommend the ones we think they'd rate highest.

### Mapping IDs to Indices

Embedding layers in PyTorch work like lookup tables — you give them an integer index, they give back a vector. But the raw user/movie IDs in our data might have gaps (user 1, 2, 5, 8...) or start from 1 instead of 0.

We need **contiguous integers starting from 0** so that every index maps to exactly one row in our embedding matrix. Think of it like assigning seat numbers: even if the original ticket numbers are scattered, we relabel them 0, 1, 2, 3... so they map neatly to rows in a matrix.

In [ ]:
# Create contiguous ID mappings: original ID → 0-based index
user_ids = ratings["user"].unique()
movie_ids = ratings["movie"].unique()

user2idx = {uid: idx for idx, uid in enumerate(user_ids)}
movie2idx = {mid: idx for idx, mid in enumerate(movie_ids)}

# Reverse mappings for interpretation later
idx2user = {idx: uid for uid, idx in user2idx.items()}
idx2movie = {idx: mid for mid, idx in movie2idx.items()}

# Also: movie index → title (for printing recommendations)
movie_id2title = dict(zip(movies["movie"], movies["title"]))
idx2title = {idx: movie_id2title[mid] for mid, idx in movie2idx.items()}

n_users = len(user2idx)
n_movies = len(movie2idx)
print(f"{n_users} users, {n_movies} movies")

# Apply mappings to the dataframe
ratings["user_idx"] = ratings["user"].map(user2idx)
ratings["movie_idx"] = ratings["movie"].map(movie2idx)
ratings.head()

### Building the DataLoader

Now we create a PyTorch `Dataset` and `DataLoader`. Each training example is a triplet: `(user_idx, movie_idx, rating)`. The DataLoader will serve these in batches of 64 — meaning each training step, we show the model 64 (user, movie) pairs and their actual ratings, compute how wrong the predictions are, and nudge the weights.

This is the same training loop pattern from L5 (neural networks) and L3 (regression) — just with different data going in.

In [ ]:
class RatingsDataset(Dataset):
    """Simple dataset: each item is (user_idx, movie_idx, rating)."""
    def __init__(self, users, movies, ratings):
        self.users = torch.LongTensor(users)
        self.movies = torch.LongTensor(movies)
        self.ratings = torch.FloatTensor(ratings)

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        return self.users[idx], self.movies[idx], self.ratings[idx]

# 80/20 train/validation split
train_df, valid_df = train_test_split(ratings, test_size=0.2, random_state=42)

train_ds = RatingsDataset(train_df["user_idx"].values, train_df["movie_idx"].values, train_df["rating"].values)
valid_ds = RatingsDataset(valid_df["user_idx"].values, valid_df["movie_idx"].values, valid_df["rating"].values)

train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)
valid_dl = DataLoader(valid_ds, batch_size=256)

# Quick check
users, movies, rats = next(iter(train_dl))
print(f"Batch shapes: users={users.shape}, movies={movies.shape}, ratings={rats.shape}")

Each batch gives us 64 user indices, 64 movie indices, and 64 ratings. The model's job: given a user index and movie index, predict the rating. That's literally the entire task.

Now before we build the model, let's develop some intuition for *what* the model is actually learning.

## Part 2: The Intuition — Latent Factors

What if each movie could be described by a few hidden qualities? Not genres you'd find on IMDB — something more nuanced. Maybe one axis captures "cerebral vs. mindless entertainment." Another captures "dark vs. lighthearted." Another might capture something humans don't even have a word for.

And what if each *user* could be described by how much they care about those same qualities?

The match between user and movie would just be a **dot product** of their vectors — multiply element-by-element, then sum. Let's see this in action with some made-up factors.

In [ ]:
# Imagine we KNEW what qualities matter.
# 3 factors: [sci-fi, action, modern]

# Last Skywalker: very sci-fi, very action, very modern
last_skywalker = np.array([0.98, 0.9, -0.9])

# A user who loves modern sci-fi action
user1 = np.array([0.9, 0.8, -0.6])

# Dot product = how well do they match?
match = (user1 * last_skywalker).sum()
print(f"User × Last Skywalker = {match:.2f}  ← high match!")

# Casablanca: not sci-fi, not action, old/classic
casablanca = np.array([-0.99, -0.3, 0.8])
match2 = (user1 * casablanca).sum()
print(f"User × Casablanca    = {match2:.2f}  ← low match")

Let's trace through what just happened. The dot product multiplies each factor pair and sums:

```
User × Last Skywalker:
  sci-fi:  0.9 × 0.98 = 0.882
  action:  0.8 × 0.9  = 0.720
  modern: -0.6 × -0.9 = 0.540
                  sum  = 2.142  ← high match!
```

Every aligned factor adds to the score. And notice: the user dislikes old movies (-0.6) and the movie is very modern (-0.9). Negative × negative = positive. The model captures that this user and this movie agree on the "modern" axis even though both values are negative.

For Casablanca, the factors *disagree* — the user likes sci-fi but Casablanca isn't, likes action but Casablanca isn't, likes modern but Casablanca is old. The dot product goes negative.

Makes sense intuitively. But we don't actually *know* what the latent factors are. We made up "sci-fi" and "action" for this demo, but real taste is more complex. Maybe there's a factor that captures "films with unreliable narrators" or "movies that make you cry on airplanes."

The breakthrough: **we don't need to define the factors.** We initialize everything randomly and let SGD figure out what factors matter by looking at which users rated which movies similarly. The model discovers the hidden structure on its own.

These factors are called **latent** (hidden) because they emerge from training — nobody defines them. After training, each number in a movie's vector *means something*, we just can't always put a label on it.

## Part 3: From One-Hot to Embedding

Here's the key question: to predict a rating, we need to "look up" a user's vector from a matrix. But how? In neural networks, everything has to be differentiable — we can't just say "grab row 3" because that's not a mathematical operation that gradients can flow through.

The trick: represent "user 3" as a **one-hot vector** — all zeros except a 1 at position 3. Then multiply that vector by the matrix. The result? You get row 3. And matrix multiplication *is* differentiable.

In [ ]:
# Create a small demo embedding matrix: 5 users, 3 factors
demo_matrix = torch.randn(5, 3)
print("Demo matrix (5 users × 3 factors):")
print(demo_matrix)

# One-hot vector for user 3: all zeros except position 3
one_hot = F.one_hot(torch.tensor(3), num_classes=5).float()
print(f"\nOne-hot for user 3: {one_hot}")

# Matrix multiply extracts row 3
result = demo_matrix.t() @ one_hot
print(f"\nMatrix multiply result: {result}")
print(f"Direct indexing result:  {demo_matrix[3]}")
print(f"Same? {torch.allclose(result, demo_matrix[3])}")

They're identical. But think about what one-hot encoding actually costs: for 943 users, each one-hot vector is 943 numbers long, with 942 of them being zero. That's a lot of wasted memory and computation.

PyTorch's `nn.Embedding` takes a shortcut: it indexes directly into the matrix (fast, memory-efficient) but computes gradients *as if* it did the full one-hot multiply. Best of both worlds.

**An embedding is a lookup table that supports gradient descent.** That's all `nn.Embedding` is — a matrix of learnable numbers where you look up a row by integer index. Under the hood, it's just an `nn.Parameter` (a matrix that PyTorch knows to update during training) plus an indexing operation.

The values start as small random numbers (drawn from a normal distribution centered at 0) and get refined by SGD as the model trains.

## Part 4: The Dot Product Model

This is the simplest collaborative filtering model. Two embedding matrices (one for users, one for movies), and a dot product to predict ratings.

But there's a subtlety: the raw dot product can output any number — negative, larger than 5, anything. Ratings are 1-5. We need to squish the output into a valid range. That's what `sigmoid_range` does: it pushes the output through a sigmoid (which produces values between 0 and 1) and then stretches it to our desired range.

We use 0 to 5.5 instead of 1 to 5 because sigmoid *asymptotically approaches* its bounds but never quite reaches them. Using 5.5 gives the model room to actually predict 5.0.

In [ ]:
def sigmoid_range(x, low, high):
    """Squish any number into the range (low, high) using sigmoid."""
    return torch.sigmoid(x) * (high - low) + low

In [ ]:
class DotProduct(nn.Module):
    def __init__(self, n_users, n_movies, n_factors, y_range=(0, 5.5)):
        super().__init__()
        # Two embedding matrices — these ARE the model's learnable parameters
        self.user_factors = nn.Embedding(n_users, n_factors)   # (943, n_factors)
        self.movie_factors = nn.Embedding(n_movies, n_factors)  # (1664, n_factors)
        self.y_range = y_range

        # Initialize with small random values
        nn.init.normal_(self.user_factors.weight, 0, 0.1)
        nn.init.normal_(self.movie_factors.weight, 0, 0.1)

    def forward(self, users, movies):
        # Look up embedding vectors for each user and movie in the batch
        user_emb = self.user_factors(users)    # (batch, n_factors)
        movie_emb = self.movie_factors(movies)  # (batch, n_factors)
        # Element-wise multiply then sum = dot product per pair
        dot = (user_emb * movie_emb).sum(dim=1)
        # Clamp to valid rating range
        return sigmoid_range(dot, *self.y_range)

That's the **entire model**. No hidden layers, no activation functions between layers. Just two embedding lookups and a dot product, clamped to the rating range.

Let's trace through what `forward` does for one pair:

```
Input: user_idx=42, movie_idx=100

1. user_emb = self.user_factors(42)    → grabs row 42 from user matrix → shape (50,)
2. movie_emb = self.movie_factors(100)  → grabs row 100 from movie matrix → shape (50,)
3. dot = (user_emb * movie_emb).sum()   → element-wise multiply, then sum → single number
4. sigmoid_range(dot, 0, 5.5)           → squish into valid rating range → e.g. 3.7
```

That's it. The only learnable parameters are the two embedding matrices — both start random. SGD's job is to adjust every number in both matrices so the dot products get closer to the actual ratings.

Now we need a training loop. In the fastai version this is a one-liner (`learn.fit()`), but here we write it ourselves so you can see every step.

In [ ]:
def train_model(model, train_dl, valid_dl, n_epochs=5, lr=5e-3, wd=0.0):
    """Train a model and return loss history."""
    model = model.to(device)
    optimizer = Adam(model.parameters(), lr=lr, weight_decay=wd)
    criterion = nn.MSELoss()

    train_losses, valid_losses = [], []

    for epoch in range(n_epochs):
        # --- Training ---
        model.train()
        batch_losses = []
        for users, movies, ratings in train_dl:
            users, movies, ratings = users.to(device), movies.to(device), ratings.to(device)

            preds = model(users, movies)          # forward pass
            loss = criterion(preds, ratings)       # MSE loss
            optimizer.zero_grad()                  # clear old gradients
            loss.backward()                        # compute new gradients
            optimizer.step()                       # update weights
            batch_losses.append(loss.item())

        # --- Validation ---
        model.eval()
        val_losses = []
        with torch.no_grad():
            for users, movies, ratings in valid_dl:
                users, movies, ratings = users.to(device), movies.to(device), ratings.to(device)
                preds = model(users, movies)
                val_losses.append(criterion(preds, ratings).item())

        train_loss = np.mean(batch_losses)
        valid_loss = np.mean(val_losses)
        train_losses.append(train_loss)
        valid_losses.append(valid_loss)
        print(f"Epoch {epoch+1}/{n_epochs}  train_loss={train_loss:.4f}  valid_loss={valid_loss:.4f}")

    return train_losses, valid_losses

The training loop follows the exact same pattern we used for regression in L3 and neural networks in L5:

1. **Forward pass**: feed data through the model to get predictions
2. **Loss**: compute how wrong the predictions are (MSE — mean squared error)
3. **Backward pass**: compute gradients (how should each weight change to reduce loss?)
4. **Update**: nudge each weight in the direction that reduces loss

The only difference from a neural network is *what* the model looks like inside. Instead of linear layers and activations, we have embedding lookups and a dot product. But SGD doesn't care — it just follows gradients.

We also track validation loss separately. Validation data is never used for training — it tells us how well the model generalizes to ratings it hasn't memorized.

In [ ]:
# Train the dot product model with 50 latent factors
n_factors = 50
model_v1 = DotProduct(n_users, n_movies, n_factors)
train_losses_v1, valid_losses_v1 = train_model(model_v1, train_dl, valid_dl, n_epochs=15, lr=1e-2)

We're using 50 latent factors — meaning each user and each movie gets described by a vector of 50 numbers. That's 50 "hidden qualities" the model can learn. Why 50? It's a common sweet spot. Too few (like 3) and the model can't capture enough nuance. Too many and it might memorize noise.

Training loss keeps dropping — the model is getting better at predicting the training data. But look at validation loss: it improves initially, then starts climbing. The model is starting to memorize training examples rather than learning general patterns. This is **overfitting**, and we'll address it soon.

First, let's visualize the loss curves, then see if adding biases helps.

In [ ]:
def plot_losses(train_losses, valid_losses, title="Training Progress"):
    plt.figure(figsize=(8, 4))
    plt.plot(train_losses, label="Train")
    plt.plot(valid_losses, label="Validation")
    plt.xlabel("Epoch")
    plt.ylabel("MSE Loss")
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

plot_losses(train_losses_v1, valid_losses_v1, "DotProduct Model")

## Part 5: Adding Biases

Think about it: some users are generous raters — they give everything 4-5 stars. Others are harsh critics — their 3 stars is actually high praise. And some movies are just universally good: even if your taste doesn't particularly align with Shawshank Redemption, you'd still rate it higher than average.

The dot product can only capture taste *match* — it multiplies user preferences with movie qualities. It has no way to say "this user tends to rate everything high" or "everyone likes this movie." Those are **baseline offsets**, not taste interactions.

Fix: add a learnable **bias** per user and per movie.

```
prediction = dot(user_vec, movie_vec) + user_bias + movie_bias
```

The user bias captures "how generous is this rater?" The movie bias captures "how good is this movie regardless of taste?"

In [ ]:
class DotProductBias(nn.Module):
    def __init__(self, n_users, n_movies, n_factors, y_range=(0, 5.5)):
        super().__init__()
        self.user_factors = nn.Embedding(n_users, n_factors)    # taste vectors
        self.user_bias = nn.Embedding(n_users, 1)               # one bias per user
        self.movie_factors = nn.Embedding(n_movies, n_factors)   # quality vectors
        self.movie_bias = nn.Embedding(n_movies, 1)              # one bias per movie
        self.y_range = y_range

        # Initialize: factors with small random values, biases at zero
        for emb in [self.user_factors, self.movie_factors]:
            nn.init.normal_(emb.weight, 0, 0.1)
        for emb in [self.user_bias, self.movie_bias]:
            nn.init.zeros_(emb.weight)

    def forward(self, users, movies):
        user_emb = self.user_factors(users)       # (batch, n_factors)
        movie_emb = self.movie_factors(movies)     # (batch, n_factors)
        # Dot product: how well does taste match?
        dot = (user_emb * movie_emb).sum(dim=1)
        # Add biases: shift based on user generosity + movie quality
        dot += self.user_bias(users).squeeze() + self.movie_bias(movies).squeeze()
        return sigmoid_range(dot, *self.y_range)

Notice the biases are `nn.Embedding(n, 1)` — one number per user/movie, looked up the same way as the factor vectors. We initialize them at zero (no bias assumed) and let SGD learn the right values.

The `.squeeze()` in the forward pass removes the extra dimension: the embedding lookup returns shape `(batch, 1)` but we need `(batch,)` to add it to the dot product scalar.

Let's train without weight decay first to see what happens. We added more parameters (the biases), which gives the model more flexibility — but also more room to overfit.

In [ ]:
model_v2 = DotProductBias(n_users, n_movies, n_factors)
train_losses_v2, valid_losses_v2 = train_model(model_v2, train_dl, valid_dl, n_epochs=15, lr=1e-2, wd=0.0)

Watch the gap between training and validation loss — training keeps dropping but validation gets worse. Classic overfitting. The extra bias parameters gave the model room to memorize training quirks instead of learning general patterns.

### Weight Decay

The fix is **weight decay** (also called L2 regularization). The idea: add a penalty to the loss function that punishes large parameter values.

```
total_loss = MSE_loss + wd × sum(all_weights²)
```

Every learnable number in the model — every value in the user embeddings, movie embeddings, user biases, movie biases — gets squared, summed up, multiplied by a small coefficient `wd`, and added to the loss.

Why does this help? Two forces now act on each parameter:
- **Prediction loss** pulls the parameter toward whatever value best predicts ratings
- **Weight decay** pulls the parameter toward zero

A parameter only stays large if it genuinely helps predictions *enough to justify the penalty*. Random memorization quirks (overfitting) don't help enough — they get shrunk toward zero. Genuinely useful patterns are strong enough to overcome the penalty.

Think of it like gravity. Useful structures (like load-bearing walls) stay up despite gravity. Flimsy structures (like a house of cards) collapse. Weight decay is gravity for parameters.

In [ ]:
# Retrain with weight decay — penalizes large embedding values
# PyTorch Adam applies weight decay differently than fastai, so we use a smaller value
model_v3 = DotProductBias(n_users, n_movies, n_factors)
train_losses_v3, valid_losses_v3 = train_model(model_v3, train_dl, valid_dl, n_epochs=15, lr=1e-2, wd=1e-5)

In [ ]:
# Compare all three models
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, tl, vl, title in zip(axes,
    [train_losses_v1, train_losses_v2, train_losses_v3],
    [valid_losses_v1, valid_losses_v2, valid_losses_v3],
    ["DotProduct", "DotProductBias (no wd)", "DotProductBias (wd=0.1)"]):
    ax.plot(tl, label="Train")
    ax.plot(vl, label="Valid")
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("MSE Loss")
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Compare the three plots. The first model (no biases) is limited — it can only capture taste match, not baseline tendencies. The second (biases, no weight decay) overfits — validation loss climbs while training loss drops. The third (biases + weight decay) strikes the best balance.

Weight decay in PyTorch Adam is set via the `weight_decay` parameter. We use a small value (`1e-5`) because PyTorch's Adam implementation applies weight decay slightly differently than some other frameworks.

## Part 6: Interpreting What the Model Learned

We started with random numbers in two big matrices. After training, those numbers mean something. The model has discovered hidden structure purely from the pattern of who rated what. Let's see what it found.

The **biases** are the easiest to interpret — each one is a single number per movie or per user. The **embedding vectors** (50 numbers each) are harder to read directly, but we can visualize them with PCA and compare them with cosine similarity.

### Movie Biases

Each movie's bias is a single number that answers: "regardless of taste match, do people tend to rate this movie higher or lower than average?" A movie with a high bias is universally loved — even users whose taste vectors don't particularly align with it still rate it high. A low bias means people generally dislike it.

Think of bias as the movie's "reputation score" that's independent of any individual's preferences.

In [ ]:
# Extract movie biases from the trained model
movie_bias = model_v3.movie_bias.weight.detach().cpu().squeeze()  # (n_movies,)

# 5 lowest biases — universally disliked
worst_idxs = movie_bias.argsort()[:5]
print("Lowest movie biases (universally disliked):")
for idx in worst_idxs:
    print(f"  {idx2title[idx.item()]:45s} bias: {movie_bias[idx]:.3f}")

# 5 highest biases — universally loved
best_idxs = movie_bias.argsort(descending=True)[:5]
print("\nHighest movie biases (universally loved):")
for idx in best_idxs:
    print(f"  {idx2title[idx.item()]:45s} bias: {movie_bias[idx]:.3f}")

The results make intuitive sense. The model figured out on its own — just from rating patterns — that Schindler's List and Shawshank Redemption are universally appreciated, while low-budget sequels like Children of the Corn: The Gathering are not.

Nobody told the model these movies are "good" or "bad." It discovered this from the statistical pattern: users who rated Schindler's List tended to rate it high regardless of their other preferences. That signal got captured in the bias term.

Now let's look at the embedding vectors, which are harder to interpret directly since each movie has 50 numbers.

### Visualizing Embeddings with PCA

Each movie has a 50-dimensional embedding vector. We can't visualize 50 dimensions, but **PCA** (Principal Component Analysis) can compress them down to 2 by finding the two directions in 50-dimensional space where the data varies the most.

Think of it like photographing a 3D sculpture. You lose depth information, but if you pick the right angle, the major structure survives. PCA picks the best "angle" for high-dimensional data. Movies that cluster together on this 2D plot have similar embedding vectors — meaning similar users liked them.

The groups you see in the plot weren't created by PCA. They were already there in the 50-dimensional embedding space. PCA just lets us see them.

In [ ]:
# Grab the top 200 most-rated movies for a cleaner plot
movie_rating_counts = ratings.groupby("movie_idx")["rating"].count()
top_movie_idxs = movie_rating_counts.sort_values(ascending=False).head(200).index.values

# Extract their learned embedding vectors
movie_embs = model_v3.movie_factors.weight.detach().cpu().numpy()  # (n_movies, 50)
top_embs = movie_embs[top_movie_idxs]  # (200, 50)

# PCA: 50 dimensions → 2
pca = PCA(n_components=2)
coords = pca.fit_transform(top_embs)

# Plot the first 50 for readability
plt.figure(figsize=(14, 14))
n_show = 50
plt.scatter(coords[:n_show, 0], coords[:n_show, 1], alpha=0.5)
for i in range(n_show):
    idx = top_movie_idxs[i]
    plt.annotate(idx2title[idx], (coords[i, 0], coords[i, 1]),
                 fontsize=9, color=np.random.RandomState(i).rand(3) * 0.7)
plt.title("Movie Embeddings (PCA → 2D)")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
plt.grid(True, alpha=0.2)
plt.show()

No matter how many times you see this, it's remarkable. The model discovered meaningful structure — genre clusters, era groupings, quality tiers — purely from rating patterns. Nobody told it what action movies or romantic comedies are. It figured out that users who like Star Wars also tend to like Raiders of the Lost Ark, and placed them near each other in embedding space.

### Finding Similar Movies with Cosine Similarity

We've seen that nearby movies in embedding space are similar. But how do we measure "nearby" precisely? **Cosine similarity** measures whether two vectors point in the same direction, regardless of how long they are.

Imagine two arrows starting from the origin. If they point the same way, cosine similarity is 1.0. If they're perpendicular (unrelated), it's 0. If they point opposite directions, it's -1.

```
Vector A →→→→→→→
Vector B →→→→→        similarity ≈ 1.0 (same direction)

Vector A →→→→→→→
Vector C ↑↑↑↑↑        similarity ≈ 0.0 (perpendicular)

Vector A →→→→→→→
Vector D ←←←←←        similarity ≈ -1.0 (opposite)
```

This is the basis for **"because you watched X"** recommendations — find movies whose embedding vectors point most similarly to the movie you just watched.

In [ ]:
# Find movies most similar to a given movie
def find_similar(title_substring, top_n=5):
    """Find the top_n most similar movies to the first movie matching title_substring."""
    # Find the movie
    matches = {idx: t for idx, t in idx2title.items() if title_substring.lower() in t.lower()}
    if not matches:
        print(f"No movie found matching '{title_substring}'")
        return
    idx = list(matches.keys())[0]
    title = matches[idx]

    # Compute cosine similarity against all movies
    movie_embs_t = model_v3.movie_factors.weight.detach().cpu()
    target = movie_embs_t[idx].unsqueeze(0)  # (1, 50)
    similarities = F.cosine_similarity(movie_embs_t, target)  # (n_movies,)

    # Sort and skip itself (similarity = 1.0)
    top_idxs = similarities.argsort(descending=True)[1:top_n+1]
    print(f"Movies most similar to '{title}':")
    for i in top_idxs:
        print(f"  {idx2title[i.item()]:45s} similarity: {similarities[i]:.3f}")

find_similar("Silence of the Lambs")
print()
find_similar("Star Wars")

Star Wars → Empire Strikes Back makes perfect sense. The model learned that the same users who love Star Wars also love its sequel, so their embeddings ended up pointing in very similar directions. Raiders of the Lost Ark and Return of the Jedi showing up is spot-on too — same era, same audience.

Some results might look odd — that's because with only 100K ratings, the embeddings aren't perfect. With millions of ratings (like Netflix or Spotify have), these similarity results get spookily accurate.

### Two Ways to Recommend

The trained embeddings support two fundamentally different recommendation strategies:

**"Top picks for you"** — Take a user's embedding vector and dot-product it with every movie's embedding. Rank by predicted score. This uses the user's *entire taste profile* — all 50 latent factors at once. It might recommend a comedy even if you just watched a thriller, because it knows your full taste.

```python
user_emb = model.user_factors(user_idx)         # (50,)
all_movies = model.movie_factors.weight          # (1664, 50)
scores = all_movies @ user_emb                   # (1664,)
top_10 = scores.argsort(descending=True)[:10]
```

**"Because you watched X"** — Take one movie's embedding and find the most similar movies using cosine similarity. This stays in the *neighborhood* of that specific film. If you just watched Star Wars, it recommends Empire Strikes Back, not a random comedy you might also like.

```python
star_wars_emb = model.movie_factors(star_wars_idx)
similarities = cosine_similarity(all_movies, star_wars_emb)
similar = similarities.argsort(descending=True)[1:6]   # skip itself
```

Production systems like Netflix use both. "Top picks for you" populates the main recommendation row. "Because you watched Stranger Things" populates a targeted row lower on the page. Different tools for different jobs, both powered by the same trained embeddings.

## Part 7: The Cold Start Problem

Everything we've built depends on one thing: trained embeddings. A user's embedding captures their taste, a movie's embedding captures its qualities. But what about a brand new user who just signed up? Or a movie that was just added to the catalog?

Their embeddings are random noise. The dot product of random noise with anything is meaningless. Let's see this in action.

In [ ]:
# Grab trained embeddings
user_embs = model_v3.user_factors.weight.detach().cpu()
movie_embs = model_v3.movie_factors.weight.detach().cpu()

# Existing user with a well-trained embedding
existing_user = user_embs[0]

# New user: random embedding (what they'd start with)
new_user = torch.randn(n_factors)

# Average user: mean of all trained embeddings
avg_user = user_embs.mean(dim=0)

def show_top5(user_vec, label):
    scores = (movie_embs * user_vec).sum(dim=1)
    top5 = scores.argsort(descending=True)[:5]
    print(f"{label}:")
    for idx in top5:
        print(f"  {idx2title[idx.item()]:45s} score: {scores[idx]:.2f}")
    print()

show_top5(existing_user, "Existing user — trained embedding")
show_top5(new_user, "New user — random embedding (garbage)")
show_top5(avg_user, "New user — average embedding (safe default)")

The random embedding produces nonsensical recommendations — obscure movies with high scores just because the random vectors happened to align. The average embedding is a safe fallback: it represents "the typical user" and gravitates toward broadly popular, well-rated films. Not personalized, but at least reasonable.

In practice, companies handle cold start with several strategies:

1. **Show popular items** as defaults — the "Trending Now" row you see before Netflix knows you
2. **Onboarding questions** — "Rate these 5 movies" gives enough signal to build a rough embedding
3. **Content features** — use genre, actors, director to bootstrap until interaction data accumulates. This is where the deep learning version (Part 8) shines: it can combine learned embeddings with known features.

**A warning about feedback loops.** If you only recommend popular items to new users, they only interact with popular items, which makes those items appear *even more* popular. A small group of very active users can disproportionately shape recommendations for everyone. Real systems need deliberate exploration (showing occasional unexpected items) to break these loops.

## Part 8: Deep Learning Version — CollabNN

Our dot product model is powerful but has two limitations:

1. **Linear only.** The dot product can capture "likes action AND likes sci-fi → high score." But it can't capture nonlinear interactions like "likes action AND comedy separately, but hates action-comedies." For that, you need nonlinearity (activation functions).

2. **No extra features.** What if we know the movie's genre, release year, or the user's age? The dot product model has no way to use that information.

The fix: instead of taking the dot product of user and movie embeddings, **concatenate** them into one long vector and feed it through a neural network. The network can learn arbitrary nonlinear relationships between the factors.

In [ ]:
class CollabNN(nn.Module):
    def __init__(self, n_users, n_movies, user_dim, movie_dim, hidden=100, y_range=(0, 5.5)):
        super().__init__()
        self.user_factors = nn.Embedding(n_users, user_dim)
        self.movie_factors = nn.Embedding(n_movies, movie_dim)
        # Neural network on top of concatenated embeddings
        self.layers = nn.Sequential(
            nn.Linear(user_dim + movie_dim, hidden),   # (user_dim + movie_dim) → hidden
            nn.ReLU(),
            nn.Linear(hidden, 1)                        # hidden → 1 predicted rating
        )
        self.y_range = y_range

        # Initialize embeddings
        nn.init.normal_(self.user_factors.weight, 0, 0.1)
        nn.init.normal_(self.movie_factors.weight, 0, 0.1)

    def forward(self, users, movies):
        user_emb = self.user_factors(users)       # (batch, user_dim)
        movie_emb = self.movie_factors(movies)     # (batch, movie_dim)
        # Concatenate instead of dot product
        x = torch.cat([user_emb, movie_emb], dim=1)  # (batch, user_dim + movie_dim)
        x = self.layers(x).squeeze()               # (batch,)
        return sigmoid_range(x, *self.y_range)

Let's trace through this model for one (user, movie) pair:

```
Input: user_idx=42, movie_idx=100

1. user_emb  = self.user_factors(42)     → shape (50,)
2. movie_emb = self.movie_factors(100)    → shape (50,)
3. x = torch.cat([user_emb, movie_emb])  → shape (100,) — one long vector
4. Linear(100 → 100) + ReLU              → shape (100,)
5. Linear(100 → 1)                       → shape (1,) — predicted rating
6. sigmoid_range(x, 0, 5.5)              → e.g. 4.2
```

The key difference from DotProduct: instead of directly multiplying the factors element-by-element (which is linear), we let a neural network *learn how to combine them*. The ReLU activation introduces nonlinearity — the model can now learn "if factor 3 is high AND factor 7 is low, that's a very different signal than either alone."

Also notice: user and movie embeddings no longer need to be the same size, since they're being concatenated rather than multiplied element-by-element.

Since we're concatenating instead of dot-producting, the user and movie embeddings don't need to be the same size. A common heuristic for embedding dimensions is `min(600, round(1.6 × n_categories^0.56))`, but for simplicity we'll keep both at 50.

We use a lower learning rate (`1e-3`) for the neural network — it has more parameters and is more sensitive to large updates.

In [ ]:
model_nn = CollabNN(n_users, n_movies, user_dim=50, movie_dim=50, hidden=100)
train_losses_nn, valid_losses_nn = train_model(model_nn, train_dl, valid_dl, n_epochs=15, lr=1e-3, wd=1e-5)

The validation loss starts lower than our dot product models — the neural network finds a better fit early on. But watch it: after a few epochs, it starts climbing again. Neural networks with more parameters are even more prone to overfitting. In a real system, you'd use techniques like dropout, more weight decay, or early stopping (saving the model from the epoch with the lowest validation loss).

In [ ]:
plot_losses(train_losses_nn, valid_losses_nn, "CollabNN (Deep Learning)")

### The Big Reveal: Collaborative Filtering IS Tabular Learning

Look at what `CollabNN` actually does: it takes two categorical inputs (user ID, movie ID), converts them to embeddings, concatenates them, and feeds them through a linear-ReLU-linear network. That's **identical** to a tabular model with only categorical features and zero continuous features.

This is the same architecture from fastai's `TabularModel` and `EmbeddingNN`. Collaborative filtering isn't some exotic technique — it's tabular ML where all your columns happen to be categorical (user ID, movie ID).

### How Production Systems Combine Everything

Real recommendation systems at Netflix, Spotify, and YouTube don't just use user and movie IDs. They concatenate the learned embeddings with **fixed features** from the dataset:

```
[user_embedding, movie_embedding, genre, year, user_age, time_of_day, ...]
                              ↓
                    Linear → ReLU → Linear → ReLU → Linear
                              ↓
                        predicted rating
```

The embeddings are **learned** — they start random and get updated by SGD every batch. They discover hidden taste patterns that can't be captured by metadata alone. The features are **fixed** — genre, year, cast. They capture known structure.

Both get concatenated into one input vector and fed through the same neural network, trained end to end. Neither alone is complete: embeddings miss explicit knowledge (genre labels), features miss hidden patterns (that users who like X also tend to like Y). Together they cover both.

**The collaborative filtering IS the embeddings.** Whether you dot-product them or feed them through a neural network, the core pattern is the same: each user and each item gets a learned vector, and similar users end up with similar vectors because that's what minimizes the loss.

## Summary

We built three progressively better recommendation models, all in pure PyTorch:

1. **DotProduct** — two embedding lookups + dot product. The simplest collaborative filtering model. Limited because it can only capture taste match, not baseline tendencies.

2. **DotProductBias** — added per-user and per-movie biases to capture "this user rates everything high" and "everyone loves Shawshank." Needed weight decay to prevent overfitting.

3. **CollabNN** — concatenated embeddings fed through a neural network. Can learn nonlinear interactions and, in production, can incorporate additional features like genre and year alongside the learned embeddings.

### Key takeaways

**Embeddings** are just lookup tables wrapped in `nn.Parameter` — matrices of learnable numbers indexed by integer ID. They start as random noise and, through training, become rich representations that capture hidden structure.

**SGD doesn't care what it's optimizing.** The same forward → loss → backward → update loop works for lines (L3), neural networks (L5), and embedding models (this lesson). The math is identical; only the model architecture changes.

**The trained embeddings are useful beyond predictions.** Cosine similarity between movie embeddings powers "because you watched X." PCA reveals hidden genre/quality structure. Biases tell you which movies are universally loved or hated.

**Collaborative filtering is tabular ML** where all your columns are categorical (user ID, movie ID). Production systems combine learned embeddings (collaborative filtering signal) with fixed features (metadata) in one neural network, trained end to end. That's how Netflix, Spotify, and YouTube actually work.

<div style="text-align: center; color: #888; font-size: 0.85em; margin-top: 40px; padding-top: 10px; border-top: 1px solid #ddd;">
© 2025 Utvecklarakademin UA Aktiebolag. All rights reserved.<br>
This material is proprietary and may not be reproduced, distributed, or shared without written permission.
</div>